# 08 · Use case: triage of affiliate interactions at a pension fund (Protección S.A.)

**Scenario.** A Colombian AFP receives large volumes of messages from affiliates (WhatsApp, web chat, email, PQRS) about very different processes: old-age pension, cesantías, regime transfers and *doble asesoría*, the pension reform, balance refunds, disability/survivor pensions, certificates, voluntary savings, account access. Each needs a different team, and some need immediate attention (complaint risk, vulnerable affiliates, customers about to leave).

**What this notebook shows.** One typed triage task answered by:
- an **LLM** (Gemini Flash-Lite here): labels, but no calibrated confidence;
- **Jev** (TypeSafe System One): the same labels in one fast pass with **calibrated probabilities**, which enables *confidence routing*: automate what is certain, send the rest to people.

**Data.** 100% **synthetic** messages generated here, with known labels. No real affiliate data. Using real data requires data-governance approval (Habeas Data, Ley 1581 de 2012, and the Superfinanciera's cloud/outsourcing requirements) and running `privacy.anonymize` first.

See `docs/usecases/proteccion_jev.md` for the business case and more ideas.

In [ ]:
%load_ext autoreload
%autoreload 2

import asyncio
import logging
import os
import random
from typing import Literal

import pandas as pd
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from pydantic_ai import Agent

from genaianalysis.eval import score
from genaianalysis.run import run_task, with_retries
from genaianalysis.schema import Conversation, Message, Task, load_jsonl, save_jsonl
from genaianalysis.utils.paths import data_processed_dir, project_dir

load_dotenv(project_dir(".env"))
os.environ.setdefault("PYDANTIC_AI_NO_BANNER", "1")
logging.basicConfig(level=logging.ERROR)
pd.set_option("display.max_colwidth", 200)

LLM = "google:gemini-flash-lite-latest"
GENERATOR = "anthropic:claude-sonnet-5" if os.getenv("ANTHROPIC_API_KEY") else LLM
RPM = 12
N_PER_INTENT = 6
MONTHLY_VOLUME = 1_000_000  # illustrative, for cost projection only

## 1 · The triage task
One schema serves both backends. `Literal` fields become Jev **Choice** questions and `bool` fields become **Noul** questions. The per-option `criteria` are the rubric: Jev receives them as option descriptions and the LLM sees them in the JSON schema. Questions are written in English (Jev's strongest language); the messages stay in Spanish.

In [ ]:
INTENTS = {
    "old_age_pension": "Requirements, status or filing of an old-age pension (pensión de vejez): age, weeks, capital, modality",
    "severance_withdrawal": "Withdrawal or balance of severance savings (cesantías): housing, education, end of employment",
    "regime_transfer": "Moving between the individual-savings regime (AFP) and the public regime (Colpensiones), or to another AFP; doble asesoría",
    "pension_reform": "Questions about the Colombian pension reform (Ley 2381 de 2024): pillars, what changes for the affiliate, transition",
    "balance_refund": "Refund of pension savings (devolución de saldos) when pension requirements are not met",
    "disability_survivor": "Disability (invalidez) or survivor (sobrevivencia) pensions, beneficiaries, medical rating",
    "certificates_statements": "Certificates, account statements (extractos), contribution history (historia laboral) or its corrections",
    "voluntary_savings": "Voluntary pension contributions (pensiones voluntarias), tax benefits, portfolios and risk profile",
    "account_access": "Password, app or website access, updating personal data",
    "other": "Anything else",
}
URGENCY = {
    "low": "Informational, no deadline",
    "medium": "Needs action soon or a process is stuck",
    "high": "Money or rights at stake now, deadline, or strong distress",
}


class Triage(BaseModel):
    intent: Literal[tuple(INTENTS)] = Field(  # type: ignore[valid-type]
        description="Which pension-fund process the affiliate's message is about",
        json_schema_extra={"criteria": INTENTS},
    )
    urgency: Literal["low", "medium", "high"] = Field(
        description="How urgently the affiliate needs a response", json_schema_extra={"criteria": URGENCY}
    )
    complaint_risk: bool = Field(description="The affiliate threatens or signals a formal complaint or escalation (queja ante la Superfinanciera, Defensor del Consumidor Financiero, tutela, demanda)")
    wants_human: bool = Field(description="The affiliate asks to talk to a person or advisor, or rejects automated answers")
    vulnerable_customer: bool = Field(description="The message shows a vulnerable situation: older adult, serious illness, disability, bereavement, or urgent economic hardship")
    transfer_risk: bool = Field(description="The affiliate says they intend to leave the fund (move to Colpensiones or another AFP) or is strongly dissatisfied with returns")


TRIAGE = Task(
    name="pension_triage",
    output_type=Triage,
    instructions="You triage incoming messages from affiliates of a Colombian pension fund (AFP). Label the main process, the urgency and the risk flags based only on what the message says.",
)

## 2 · Synthetic interactions with known labels

In [ ]:
class _Msg(BaseModel):
    text: str


GEN_PROMPT = """Escribe un mensaje realista de un afiliado a un fondo de pensiones colombiano, enviado por {channel}.
Tema: {intent_desc}.
Urgencia: {urgency} ({urgency_desc}).
{flags}
Español colombiano natural, 1 a 4 frases, puede tener errores de digitación. Inventa nombres y números falsos. No nombres empresas reales (ni fondos, ni bancos).
No menciones etiquetas como "urgencia alta" ni el nombre del tema de forma literal."""
FLAG_TEXT = {
    "complaint_risk": "El afiliado amenaza con poner una queja formal (Superfinanciera, Defensor del Consumidor Financiero, tutela o demanda).",
    "wants_human": "El afiliado pide hablar con una persona o asesor.",
    "vulnerable_customer": "El afiliado está en una situación vulnerable (adulto mayor, enfermedad, discapacidad, duelo o apuro económico).",
    "transfer_risk": "El afiliado dice que se quiere ir del fondo (a Colpensiones u otra AFP) o está muy inconforme con los rendimientos.",
}
FLAGS = list(FLAG_TEXT)


async def generate(n_per_intent: int, seed: int = 0) -> list[Conversation]:
    rng = random.Random(seed)
    agent = Agent(GENERATOR, output_type=_Msg)
    specs = [
        {"intent": i, "urgency": rng.choice(list(URGENCY)), "channel": rng.choice(["WhatsApp", "chat web", "correo"]),
         **{f: rng.random() < 0.25 for f in FLAGS}}
        for i in INTENTS for _ in range(n_per_intent)
    ]

    async def one(k: int, s: dict) -> Conversation | None:
        await asyncio.sleep(k * 60 / RPM)
        flags = " ".join(FLAG_TEXT[f] for f in FLAGS if s[f]) or "Sin señales de queja, vulnerabilidad ni intención de retiro."
        prompt = GEN_PROMPT.format(channel=s["channel"], intent_desc=INTENTS[s["intent"]], urgency=s["urgency"], urgency_desc=URGENCY[s["urgency"]], flags=flags)
        try:
            r = await with_retries(lambda: agent.run(prompt, model_settings={"temperature": 1.0}))
        except Exception as e:  # keep going; failed items are skipped
            print("skip", k, type(e).__name__)
            return None
        return Conversation(id=f"afiliado-{k}", messages=[Message(sender="afiliado", text=r.output.text)],
                            labels={k2: str(v) for k2, v in s.items() if k2 != "channel"}, meta={"channel": s["channel"]})

    return [c for c in await asyncio.gather(*(one(k, s) for k, s in enumerate(specs))) if c]


path = data_processed_dir(f"pension_triage_{N_PER_INTENT}.jsonl")
if not path.exists():
    save_jsonl(await generate(N_PER_INTENT), path)
msgs = load_jsonl(path)
print(len(msgs), "messages")
pd.DataFrame([{"text": c.messages[0].text, **c.labels} for c in msgs]).sample(5, random_state=1)

## 3 · LLM baseline

In [ ]:
llm = await run_task(TRIAGE, msgs, LLM, rpm=RPM)
FIELDS = ["intent", "urgency", *FLAGS]
llm_scores = pd.concat([score(llm, msgs, f).assign(field=f) for f in FIELDS])
llm_scores[["field", "model", "coverage", "accuracy", "macro_f1", "kappa", "latency_p50_s", "cost_usd_per_1k"]]

## 4 · Jev: same task, calibrated confidence
Needs `TYPESAFE_API_KEY`. Without it, the rest of the notebook uses the LLM only.

In [ ]:
HAS_JEV = bool(os.getenv("TYPESAFE_API_KEY"))
if HAS_JEV:
    from genaianalysis.backends.jev import run_jev

    jev = await run_jev(TRIAGE, msgs)
    both = pd.concat([score(pd.concat([jev, llm]), msgs, f).assign(field=f) for f in FIELDS])
    display(both[["field", "backend", "accuracy", "macro_f1", "latency_p50_s", "cost_usd_per_1k", "ece"]])
else:
    print("TYPESAFE_API_KEY not set; skipping Jev")

## 5 · Confidence routing: how much can be automated safely?
Messages whose intent confidence is ≥ *t* are routed automatically; the rest go to a person (or an LLM). Leaders choose *t* from this curve: automation rate vs accuracy of what is automated.

In [ ]:
if HAS_JEV:
    gold = {c.id: c.labels["intent"] for c in msgs}
    j = jev.dropna(subset=["intent"]).assign(ok=lambda d: d["id"].map(gold) == d["intent"])
    rows = []
    for t in [i / 20 for i in range(21)]:
        auto = j[j["intent_confidence"] >= t]
        rows.append({"threshold": t, "automated": len(auto) / len(j), "accuracy_automated": auto["ok"].mean() if len(auto) else None})
    curve = pd.DataFrame(rows)
    ax = curve.plot(x="automated", y="accuracy_automated", marker="o", figsize=(6, 4), legend=False,
                    xlabel="share of messages routed automatically", ylabel="intent accuracy on those", title="Jev confidence routing")
    display(curve.iloc[::4])

## 6 · What a team lead would see: the queue

In [ ]:
view = (jev if HAS_JEV else llm).dropna(subset=["intent"])
queue = pd.crosstab(view["intent"], view["urgency"]).reindex(columns=["high", "medium", "low"], fill_value=0)
queue["total"] = queue.sum(axis=1)
display(queue.sort_values("high", ascending=False))

text = {c.id: c.messages[0].text for c in msgs}
alerts = view[view["complaint_risk"] | view["vulnerable_customer"] | view["transfer_risk"]]
alerts.assign(text=alerts["id"].map(text))[["text", "intent", "urgency", "complaint_risk", "vulnerable_customer", "transfer_risk", "wants_human"]].head(10)

## 7 · Cost at scale (illustrative)
Measured cost per message × an assumed monthly volume. Jev bills input tokens only ($0.042 per 1M), and all questions share one read of the message.

In [ ]:
frames = [llm] + ([jev] if HAS_JEV else [])
pd.DataFrame([
    {"backend": f["backend"].iloc[0], "model": f["model"].iloc[0],
     "usd_per_message": f["cost_usd"].mean(), "p50_latency_s": f["latency_s"].median(),
     f"usd_per_{MONTHLY_VOLUME:,}_messages": f["cost_usd"].mean() * MONTHLY_VOLUME}
    for f in frames
])

## Takeaways to record
Per field: accuracy LLM vs Jev (Spanish input), ECE, the threshold that automates the most at ≥95% intent accuracy, and cost per million messages. Add them to `docs/research.md` and the use-case doc.